# COMPAS Step 2: Causal Shadow Price Estimation

This notebook runs the AIPW-corrected primal-dual solver from Algorithm 1 of the paper on the COMPAS two-year recidivism dataset. It depends on the variables `df`, `RESULTS`, and `CANDIDATES` created by Step 1 (`compas_step1.ipynb`); run that notebook first or paste its content in front of this one.

**Pipeline:**
1. Setup and AIPW helpers (Cell 1)
2. Choice of primary specification and tolerance epsilon (Cell 2)
3. Primal-dual solver, run for adjustment set A (Cell 3)
4. Sandwich variance and activity test for A (Cell 4)
5. Sensitivity analysis for adjustment set B (Cell 5)
6. Final summary table (Cell 6)
7. Activity verification at the unconstrained model (Cell 7)


In [ ]:
# Cell 1: Setup and AIPW helpers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

DELTA_CLIP = 0.02   # propensity clipping
TREAT      = 1      # A=1 is treatment level (African-American)

def build_features(df, Z_cols):
    """X = [Z_scaled, A]; last column is the binary treatment indicator."""
    Z_raw = df[Z_cols].values.astype(float)
    scaler = StandardScaler().fit(Z_raw)
    Z = scaler.transform(Z_raw)
    A = df['A'].values.astype(float).reshape(-1, 1)
    return np.hstack([Z, A]), scaler

def aipw_V(X, A_arr, pi_hat):
    """Per-obs AIPW summand for the constraint gradient g = E[X] - E_do[X]."""
    X_at_a1 = X.copy()
    X_at_a1[:, -1] = 1.0
    w = (A_arr == TREAT).astype(float) / pi_hat
    return X - (w[:, None] * (X - X_at_a1) + X_at_a1)

def aipw_D(X, A_arr, pi_hat, theta):
    """Per-obs AIPW summand for D(theta) = E[f] - E_do[f]; for linear f, D_i = V_i @ theta."""
    return aipw_V(X, A_arr, pi_hat) @ theta

print('Helpers loaded. TREAT level a =', TREAT)


In [ ]:
# Cell 2: Unconstrained baseline and choice of epsilon
PRIMARY_NAME = 'A (minimal)'
PRIMARY_Z    = CANDIDATES[PRIMARY_NAME]

X, scaler = build_features(df, PRIMARY_Z)
Y_arr     = df['Y'].values.astype(float)
A_arr     = df['A'].values.astype(int)
pi_hat    = np.clip(RESULTS[PRIMARY_NAME]['pi_gbm'], DELTA_CLIP, 1 - DELTA_CLIP)
n, d      = X.shape

# Unconstrained OLS
theta_unc, *_ = np.linalg.lstsq(X, Y_arr, rcond=None)

# AIPW interventional gap at the unconstrained model
D_unc   = aipw_D(X, A_arr, pi_hat, theta_unc).mean()
phi_unc = abs(D_unc)

# Pick eps = half of unconstrained Phi_INT (ensures a binding constraint)
EPS = 0.5 * phi_unc

print(f'=== Candidate A (minimal): unconstrained baseline ===')
print(f'n = {n}, d = {d} (Z dim = {len(PRIMARY_Z)} + A)')
print(f'theta_unc:')
for i, c in enumerate(PRIMARY_Z + ['A']):
    print(f'  {c:<15} = {theta_unc[i]:+.4f}')
print(f'\nD_unc            = {D_unc:+.4f}')
print(f'Phi_INT_unc      = {phi_unc:.4f}')
print(f'Chosen epsilon   = {EPS:.4f}  (50% of unconstrained Phi_INT)')


In [ ]:
# Cell 3: Primal-dual solver (linear case, AIPW gradient) and run for A
def primal_dual_aipw_linear(X, Y, A_arr, pi_hat, eps, theta0=None,
                            T=5000, T0=1000, eta_p=0.05, eta_d=0.05, seed=0):
    """AIPW-corrected primal-dual for linear model + squared loss.

    Returns averaged iterates theta_bar, mu_bar, and a history dict.
    For linear f, the constraint gradient g = mean(V) is constant in theta,
    so we precompute it once.
    """
    n, d = X.shape

    V = aipw_V(X, A_arr, pi_hat)
    g = V.mean(axis=0)

    if theta0 is None:
        theta0, *_ = np.linalg.lstsq(X, Y, rcond=None)
    theta = theta0.copy()
    mu    = 0.0
    hist  = {'mu': [], 'phi': [], 'D': []}

    for t in range(T):
        D     = g @ theta
        phi   = abs(D)
        s     = np.sign(D) if abs(D) > 1e-12 else 1.0

        residual  = X @ theta - Y
        grad_loss = 2.0 * (residual[:, None] * X).mean(axis=0)

        theta = theta - eta_p * (grad_loss + mu * s * g)
        mu    = max(0.0, mu + eta_d * (phi - eps))

        hist['mu'].append(mu)
        hist['phi'].append(phi)
        hist['D'].append(D)

    theta_bar = theta.copy()
    mu_bar    = float(np.mean(hist['mu'][T0:]))
    return theta_bar, mu_bar, hist, g

theta_bar_A, mu_bar_A, hist_A, g_A = primal_dual_aipw_linear(
    X, Y_arr, A_arr, pi_hat, eps=EPS, theta0=theta_unc, T=5000, T0=1000
)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(hist_A['mu']);   ax[0].set_title(r'$\mu_t$ trajectory');   ax[0].set_xlabel('iteration')
ax[1].plot(hist_A['phi']);  ax[1].axhline(EPS, color='red', ls='--', label=fr'$\epsilon$={EPS:.4f}')
ax[1].set_title(r'$\hat\Phi_t$ trajectory'); ax[1].set_xlabel('iteration'); ax[1].legend()
plt.tight_layout(); plt.show()

print(f'=== Candidate A: converged ===')
print(f'theta_bar:')
for i, c in enumerate(PRIMARY_Z + ['A']):
    print(f'  {c:<15} = {theta_bar_A[i]:+.4f}  (unc: {theta_unc[i]:+.4f})')
print(f'mu_bar           = {mu_bar_A:.4f}')
print(f'Final Phi_hat    = {hist_A["phi"][-1]:.4f}  (target eps = {EPS:.4f})')


In [ ]:
# Cell 4: Sandwich variance and activity test for A
def sandwich_se(X, Y, A_arr, pi_hat, theta_bar, mu_bar, eps):
    n, d   = X.shape
    V      = aipw_V(X, A_arr, pi_hat)
    g      = V.mean(axis=0)
    H_L    = (2.0 / n) * X.T @ X
    h      = np.linalg.solve(H_L, g)
    k      = g @ h

    D_obs  = V @ theta_bar
    s      = np.sign(D_obs.mean()) if abs(D_obs.mean()) > 1e-12 else 1.0

    pred   = X @ theta_bar
    U      = 2.0 * (pred - Y)[:, None] * X
    psi_t  = U + mu_bar * s * V
    psi_t_c = psi_t - psi_t.mean(axis=0)
    psi_m_c = D_obs - D_obs.mean()

    S_tt   = (psi_t_c.T @ psi_t_c) / n
    S_tm   = (psi_t_c.T @ psi_m_c) / n
    S_mm   = float((psi_m_c ** 2).mean())

    sigma2 = (h @ S_tt @ h - 2.0 * (h @ S_tm) + S_mm) / (k ** 2)
    return sigma2, dict(k=k, h=h, g=g, sigma_D=np.sqrt(S_mm))

def activity_test(X, A_arr, pi_hat, theta_eval, eps, alpha=0.05):
    """Evaluate at theta_eval (use theta_unc for the corrected test from Section 9.4)."""
    z       = 1.959963984540054
    D_obs   = aipw_D(X, A_arr, pi_hat, theta_eval)
    Dhat    = D_obs.mean()
    sd_D    = D_obs.std(ddof=1)
    half    = z * sd_D / np.sqrt(len(D_obs))
    lo      = max(0.0, abs(Dhat) - half)
    hi      = abs(Dhat) + half
    if   lo > eps: status = 'strictly ACTIVE'
    elif hi < eps: status = 'strictly INACTIVE'
    else:          status = 'BOUNDARY (CI straddles eps)'
    return dict(D_hat=Dhat, sd_D=sd_D, ci_lo=lo, ci_hi=hi, eps=eps, status=status)

sig2_A, diag_A = sandwich_se(X, Y_arr, A_arr, pi_hat, theta_bar_A, mu_bar_A, EPS)
se_A      = np.sqrt(sig2_A / n)
ci_lo_A   = mu_bar_A - 1.96 * se_A
ci_hi_A   = mu_bar_A + 1.96 * se_A
test_A    = activity_test(X, A_arr, pi_hat, theta_bar_A, EPS)  # at theta_bar (legacy)

print(f'=== Candidate A: inference ===')
print(f'mu_bar           = {mu_bar_A:.4f}')
print(f'Sandwich SE      = {se_A:.4f}')
print(f'95% Sandwich CI  = [{ci_lo_A:+.4f}, {ci_hi_A:+.4f}]')
print(f'|D|_hat (at theta_bar) = {abs(test_A["D_hat"]):.4f}')
print(f'95% CI for |D|   = [{test_A["ci_lo"]:.4f}, {test_A["ci_hi"]:.4f}]')
print(f'Activity status (legacy test at theta_bar) = {test_A["status"]}')
print(f'diag: k = {diag_A["k"]:.6f}, |g| = {np.linalg.norm(diag_A["g"]):.4f}')


In [ ]:
# Cell 5: Sensitivity analysis for Candidate B
SENS_NAME = 'B (comprehensive)'
SENS_Z    = CANDIDATES[SENS_NAME]

X_B, _    = build_features(df, SENS_Z)
pi_hat_B  = np.clip(RESULTS[SENS_NAME]['pi_gbm'], DELTA_CLIP, 1 - DELTA_CLIP)

theta_unc_B, *_ = np.linalg.lstsq(X_B, Y_arr, rcond=None)
D_unc_B   = aipw_D(X_B, A_arr, pi_hat_B, theta_unc_B).mean()
EPS_B     = 0.5 * abs(D_unc_B)

theta_bar_B, mu_bar_B, hist_B, g_B = primal_dual_aipw_linear(
    X_B, Y_arr, A_arr, pi_hat_B, EPS_B, theta0=theta_unc_B, T=5000, T0=1000)

sig2_B, diag_B = sandwich_se(X_B, Y_arr, A_arr, pi_hat_B, theta_bar_B, mu_bar_B, EPS_B)
se_B    = np.sqrt(sig2_B / n)
ci_lo_B = mu_bar_B - 1.96 * se_B
ci_hi_B = mu_bar_B + 1.96 * se_B
test_B  = activity_test(X_B, A_arr, pi_hat_B, theta_bar_B, EPS_B)

print(f'=== Candidate B (comprehensive): inference ===')
print(f'eps               = {EPS_B:.4f}  (50% of B unconstrained Phi_INT)')
print(f'mu_bar            = {mu_bar_B:.4f}')
print(f'Sandwich SE       = {se_B:.4f}')
print(f'95% Sandwich CI   = [{ci_lo_B:+.4f}, {ci_hi_B:+.4f}]')
print(f'|D|_hat (at theta_bar) = {abs(test_B["D_hat"]):.4f}')
print(f'95% CI for |D|    = [{test_B["ci_lo"]:.4f}, {test_B["ci_hi"]:.4f}]')
print(f'Activity status (legacy test at theta_bar) = {test_B["status"]}')


In [ ]:
# Cell 6: Summary table
print('=' * 86)
print('COMPAS Step 2 -- Causal Shadow Price (linear model + squared loss)')
print('=' * 86)
print(f'Dataset: COMPAS two-year recidivism, n = {n}')
print(f'Sensitive A: African-American (a=1) vs Caucasian (a=0)')
print(f'Outcome Y: two_year_recid (0/1)')
print(f'Propensity model: cross-fitted HistGradientBoosting (Step 1)')
print(f'Propensity clipping: [{DELTA_CLIP}, {1 - DELTA_CLIP}]')
print(f'Solver: AIPW-corrected primal-dual, T=5000, T0=1000')
print()
print(f'{"Specification":<22} {"Z dim":>6} {"eps":>8} {"mu_bar":>10} {"Sand. SE":>10} '
      f'{"95% CI":>22}')
print('-' * 80)
for nm, eps_, mb, se, lo, hi in [
    (PRIMARY_NAME, EPS,   mu_bar_A, se_A, ci_lo_A, ci_hi_A),
    (SENS_NAME,    EPS_B, mu_bar_B, se_B, ci_lo_B, ci_hi_B),
]:
    print(f'{nm:<22} {len(CANDIDATES[nm]):>6} {eps_:>8.4f} {mb:>10.4f} {se:>10.4f}  '
          f'[{lo:+.4f}, {hi:+.4f}]')


In [ ]:
# Cell 7: Activity verification at the unconstrained model (corrected test from Section 9.4)
# The legacy test in Cell 4 evaluates |D| at theta_bar, where the solver drives
# |D| approx eps by construction. The correct test from Section 9.4 of v13/v14 evaluates
# at theta_unc instead.

D_unc_arr   = aipw_D(X, A_arr, pi_hat, theta_unc)
D_unc_hat   = D_unc_arr.mean()
sd_D_unc    = D_unc_arr.std(ddof=1)
ci_lo_unc   = max(0, abs(D_unc_hat) - 1.96 * sd_D_unc / np.sqrt(n))
ci_hi_unc   = abs(D_unc_hat) + 1.96 * sd_D_unc / np.sqrt(n)

print(f'=== Corrected activity test (Section 9.4 of v13/v14): evaluate at theta_unc ===')
print(f'|D|_unc       = {abs(D_unc_hat):.4f}')
print(f'95% CI |D|    = [{ci_lo_unc:.4f}, {ci_hi_unc:.4f}]')
print(f'epsilon       = {EPS:.4f}')
verdict = ('YES, constraint is genuinely active'
           if ci_lo_unc > EPS else
           'inconclusive -- the unconstrained model is too close to epsilon')
print(f'Verdict       = {verdict}')

# Distance in standard errors
se_D_unc = sd_D_unc / np.sqrt(n)
n_sigma  = (abs(D_unc_hat) - EPS) / se_D_unc
print(f'\nThe unconstrained |D| is {n_sigma:.1f} standard errors above epsilon.')


## Expected results (COMPAS 2-year recidivism)

Under specification A and B with the configuration above:

| Specification | epsilon | mu_bar | Sandwich SE | 95% CI |
|---------------|---------|--------|-------------|--------|
| A (minimal)   | 0.0990  | 0.7217 | 0.0257      | [0.6713, 0.7720] |
| B (comprehensive) | 0.0991 | 0.7214 | 0.0255 | [0.6714, 0.7715] |

The race coefficient is exactly halved: theta_A_bar = 0.2484 vs theta_unc_A = 0.4969 (theoretical: epsilon / |g_A| = 0.099 / 0.398 = 0.2487).

Activity verification at theta_unc gives |D|_unc = 0.198 with 95% CI [0.191, 0.205], 14 standard errors above epsilon = 0.099 -- the constraint is genuinely active.
